# 🚀 Data Engineering Pipeline Tutorial - Part 3
## Transformation, dbt & Incremental Loading

### What You'll Learn:
1. Loading data to PostgreSQL warehouse
2. **dbt** for SQL transformations
3. **Incremental loading** - only process new data
4. Pipeline orchestration patterns

---
## 📥 Part 6: Loading to PostgreSQL

Key concept: **TRUNCATE vs DROP**
- DROP removes table and breaks dependent views
- TRUNCATE clears data but keeps structure and dependencies

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

POSTGRES_CONFIG = {
    "host": "postgres",
    "port": 5432,
    "database": "devdb",
    "user": "devuser",
    "password": "devpassword"
}

class PostgresLoader:
    def __init__(self, config):
        self.conn = psycopg2.connect(**config)
    
    def load(self, df, table_name, if_exists='replace'):
        """
        Load DataFrame to PostgreSQL.
        
        if_exists='replace' uses TRUNCATE (not DROP)
        This preserves dbt views that depend on the table!
        """
        if if_exists == 'replace':
            with self.conn.cursor() as cur:
                try:
                    cur.execute(f"TRUNCATE TABLE {table_name}")
                except:
                    pass  # Table doesn't exist yet
            self.conn.commit()
        
        # Batch insert for performance
        columns = ', '.join([f'"{c}"' for c in df.columns])
        values = [tuple(row) for row in df.values]
        
        with self.conn.cursor() as cur:
            execute_values(cur, f"INSERT INTO {table_name} ({columns}) VALUES %s", values)
        self.conn.commit()
        
        print(f"✅ Loaded {len(df)} rows to {table_name}")

---
## 🔧 Part 7: dbt (Data Build Tool)

**dbt** transforms data using SQL, with:
- Version control (git)
- Testing
- Documentation
- Incremental loading

### Our dbt Project Structure:
```
dbt_project/
├── models/
│   ├── staging/           # Clean raw data (VIEWS)
│   │   ├── stg_customers.sql
│   │   └── stg_orders.sql
│   └── marts/             # Business tables (INCREMENTAL)
│       ├── dim_customers.sql
│       └── fact_orders.sql
└── dbt_project.yml
```

### Staging Model (View)

```sql
-- models/staging/stg_customers.sql
-- This creates a VIEW - always reflects latest data

SELECT
    customer_id,
    TRIM(name) as name,
    LOWER(TRIM(email)) as email,
    TRIM(city) as city,
    _extracted_at
FROM {{ source('raw', 'mysql_customers') }}
WHERE customer_id IS NOT NULL
```

**Why VIEW?** 
- No data duplication
- Always current
- Fast to create

### Incremental Model (Mart)

```sql
-- models/marts/dim_customers.sql
-- INCREMENTAL: Only process NEW records

{{
    config(
        materialized='incremental',
        unique_key='customer_id'
    )
}}

SELECT
    customer_id,
    name,
    email,
    city,
    _extracted_at as last_updated
FROM {{ ref('stg_customers') }}

{% if is_incremental() %}
    -- Only get records newer than what we have
    WHERE _extracted_at > (SELECT MAX(last_updated) FROM {{ this }})
{% endif %}
```

**How it works:**
1. First run: Creates table, loads ALL data
2. Subsequent runs: Only inserts NEW records (based on timestamp)
3. `unique_key`: If duplicate found, UPDATE instead of INSERT

### Running dbt

```bash
# Run all models
./run_in_docker.sh dbt

# What happens:
# 1. stg_customers (VIEW) - created/replaced
# 2. stg_orders (VIEW) - created/replaced  
# 3. dim_customers (INCREMENTAL) - new rows merged
# 4. fact_orders (INCREMENTAL) - new rows appended
```

---
## 🔄 Part 8: Incremental Data Flow

### The Daily Workflow

```
Day 1 (Initial Load):
MySQL: 5 customers, 7 orders
  ↓ pipeline.py
Warehouse: 5 customers, 7 orders
  ↓ dbt run
dim_customers: 5 rows
fact_orders: 7 rows

Day 2 (Delta):
MySQL: 7 customers (+2), 11 orders (+4)
  ↓ pipeline.py (extracts all, loads fresh)
Warehouse: 7 customers, 11 orders
  ↓ dbt run (incremental!)
dim_customers: 7 rows (merged 2 new)
fact_orders: 11 rows (appended 4 new)
```

In [ ]:
# Verify the data flow
conn = psycopg2.connect(**POSTGRES_CONFIG)

tables = ['mysql_customers', 'mysql_orders', 'dim_customers', 'fact_orders']

print("📊 Current Record Counts:")
for table in tables:
    try:
        df = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
        print(f"   {table}: {df['cnt'].values[0]} rows")
    except:
        print(f"   {table}: (not created yet)")

conn.close()

---
## 🎯 Part 9: Key Patterns Summary

| Pattern | What | Why |
|---------|------|-----|
| **Watermark** | Track `_extracted_at` | Know what's new |
| **Incremental** | Only process delta | Faster, cheaper |
| **Idempotent** | Re-run = same result | Safe retries |
| **TRUNCATE** | Clear without DROP | Preserve dependencies |
| **unique_key** | Merge on conflict | Handle updates |

### Commands Cheatsheet

```bash
# Generate test data
./run_in_docker.sh generate

# Run ETL pipeline
./run_in_docker.sh pipeline

# Run dbt transforms
./run_in_docker.sh dbt

# Verify counts
./run_in_docker.sh verify

# Full cycle (all above)
./run_in_docker.sh full
```

---
## 🏆 Complete Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  generate_delta.py → MySQL (source)                         │
└─────────────────────────────────────────────────────────────┘
                              │
                     pipeline.py (Extract)
                              │
                              ▼
┌─────────────────────────────────────────────────────────────┐
│  MinIO: bronze/ → silver/ → gold/                           │
└─────────────────────────────────────────────────────────────┘
                              │
                     pipeline.py (Load)
                              │
                              ▼
┌─────────────────────────────────────────────────────────────┐
│  PostgreSQL: mysql_customers, mysql_orders (raw)            │
└─────────────────────────────────────────────────────────────┘
                              │
                         dbt run
                              │
                              ▼
┌─────────────────────────────────────────────────────────────┐
│  PostgreSQL: dim_customers, fact_orders (marts)             │
└─────────────────────────────────────────────────────────────┘
```

**You now have a real-world data engineering pipeline!** 🎉